In [1]:
# 🎬 🎓 SCENARIO: “College Smart Assistant System”
# 🏫 Background Story

# A college builds an AI-powered student assistant.

# 👉 Students can ask:

# “What is my attendance?”
# “What are my marks?”

# 👉 Instead of manually checking portals,
# 👉 AI fetches it instantly.
# ================================
# STEP 1: Install Gradio
# ================================
!pip install gradio


# ================================
# STEP 2: Dummy Database (Tool Layer)
# ================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}


# ================================
# STEP 3: Tool Functions (MCP Tools)
# ================================
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"


def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"


# ================================
# STEP 4: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id


# ================================
# STEP 5: MCP Agent Logic
# ================================
def mcp_agent(message, student_id, history):

    # Simulate logged-in user (for demo)
    user_id = student_id

    # Security Check
    if not secure_access(user_id, student_id):
        return "🚫 Access Denied", history

    # Tool Invocation Logic
    message_lower = message.lower()

    if "attendance" in message_lower:
        response = get_attendance(student_id)

    elif "marks" in message_lower:
        response = get_marks(student_id)

    elif "hello" in message_lower or "hi" in message_lower:
        response = "👋 Hello! Ask me about attendance or marks."

    else:
        response = "🤖 I can help with attendance or marks."

    # Maintain chat history
    history.append((message, response))

    return "", history


# ================================
# STEP 6: Gradio Chat UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🎓 Student MCP Agent (Gradio Version)")

    student_id = gr.Textbox(label="Enter Student ID (e.g., 101)")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[msg, chatbot]
    )

# ================================
# STEP 7: Launch App (Public URL)
# ================================
demo.launch(share=True)

Defaulting to user installation because normal site-packages is not writeable
  Using cached fastapi-0.135.2-py3-none-any.whl.metadata (28 kB)
  Using cached uvicorn-0.42.0-py3-none-any.whl.metadata (6.7 kB)
Using cached fastapi-0.135.2-py3-none-any.whl (117 kB)
Using cached uvicorn-0.42.0-py3-none-any.whl (68 kB)

   ---------------------------------------- 0/2 [uvicorn]
   -------------------- ------------------- 1/2 [fastapi]
   -------------------- ------------------- 1/2 [fastapi]
   ---------------------------------------- 2/2 [fastapi]



C:\Users\sk165\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [2]:
# ======================================
# STEP 1: Install Libraries
# ======================================
!pip install groq gradio


# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Dummy Database (Tool Layer)
# ======================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}


# ======================================
# STEP 4: Tool Functions
# ======================================
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"


def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"


# ======================================
# STEP 5: MCP Tool Decision via LLM
# ======================================
def decide_tool(query):
    try:
        prompt = f"""
        You are an AI assistant.

        Decide which function to call:
        - get_attendance
        - get_marks

        Rules:
        - If user asks about attendance → get_attendance
        - If user asks about marks → get_marks

        Only return function name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        tool = response.choices[0].message.content.strip().lower()

        return tool

    except Exception as e:
        print("❌ Groq Error:", e)
        return "fallback"


# ======================================
# STEP 6: MCP Agent (CORE LOGIC)
# ======================================
def mcp_agent(message, student_id, history):

    # Validate input
    if not student_id:
        response = "⚠️ Please enter Student ID"
        history.append((message, response))
        return "", history

    # Step 1: LLM decides tool
    tool = decide_tool(message)

    # Step 2: Tool Invocation
    if "attendance" in tool:
        response = get_attendance(student_id)

    elif "marks" in tool:
        response = get_marks(student_id)

    # Fallback (if LLM fails)
    elif tool == "fallback":
        if "attendance" in message.lower():
            response = get_attendance(student_id)
        elif "marks" in message.lower():
            response = get_marks(student_id)
        else:
            response = "⚠️ LLM failed, and I couldn't understand."

    else:
        response = "🤖 I can help with attendance or marks."

    # Step 3: Save chat
    history.append((message, response))

    return "", history


# ======================================
# STEP 7: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🚀 MCP Agent with Groq (Stable Version)")

    student_id = gr.Textbox(label="Enter Student ID (101 / 102)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[msg, chatbot]
    )


# ======================================
# STEP 8: Launch App
# ======================================
demo.launch(share=True)

Defaulting to user installation because normal site-packages is not writeable


ModuleNotFoundError: No module named 'google.colab'

Traceback (most recent call last):
  File "C:\Users\sk165\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\sk165\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "C:\Users\sk165\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\gradio\blocks.py", line 2169, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^

In [5]:
# SCENARIO: “AI Banking Assistant with Role-Based Access”
# 🏦 Background Story

# A bank builds an AI assistant to help users:

# Check account balance
# View transactions
# Approve loans
# Manage customer accounts

# 👉 But not everyone can do everything

# 🧑‍🤝‍🧑 Roles in the Bank
# Role	Description
# 👤 Customer	Bank account holder
# 👨‍💼 Employee	Bank staff
# 🧑‍💼 Manager	Branch manager
# 🔐 Permissions (RBAC)
# Action	Customer	Employee	Manager
# View own balance	✅	✅	✅
# View others' accounts	❌	✅	✅
# Approve loan	❌	❌	✅
# View all transactions	❌	✅	✅


# ======================================
# STEP 1: Install Libraries
# ======================================
# !pip install groq gradio


# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("❌ GROQ_API_KEY not found in Colab Secrets")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Dummy Bank Database
# ======================================
accounts = {
    "1001": {"name": "Amit", "balance": 50000},
    "1002": {"name": "Neha", "balance": 75000}
}


# ======================================
# STEP 4: Tool Functions (APIs)
# ======================================
def get_balance(account_id):
    if account_id in accounts:
        return f"💰 Balance of {account_id}: ₹{accounts[account_id]['balance']}"
    return "❌ Account not found"


def approve_loan(account_id):
    if account_id in accounts:
        return f"🏦 Loan approved for account {account_id}"
    return "❌ Account not found"


# ======================================
# STEP 5: RBAC Security Layer
# ======================================
def secure_access(role, user_account, requested_account, action):

    # Manager → full access
    if role == "manager":
        return True

    # Employee → can view all but cannot approve loans
    elif role == "employee":
        if action == "approve_loan":
            return False
        return True

    # Customer → only own account, no loan approval
    elif role == "customer":
        return user_account == requested_account and action != "approve_loan"

    return False


# ======================================
# STEP 6: MCP Tool Decision via LLM
# ======================================
def decide_action(query):
    try:
        prompt = f"""
        You are an AI banking assistant.

        Decide which action to take:
        - get_balance
        - approve_loan

        Rules:
        - Balance related queries → get_balance
        - Loan approval queries → approve_loan

        Return ONLY the action name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        action = response.choices[0].message.content.strip().lower()
        return action

    except Exception as e:
        print("❌ Groq Error:", e)
        return "fallback"


# ======================================
# STEP 7: MCP Agent (Core Logic)
# ======================================
def banking_agent(message, role, user_account, requested_account, history):

    # Input validation
    if not user_account:
        response = "⚠️ Please enter your account ID"
        history.append((message, response))
        return "", history

    if not requested_account:
        requested_account = user_account  # default to own account

    # Step 1: LLM decides action
    action = decide_action(message)

    # Step 2: RBAC Security Check
    if not secure_access(role, user_account, requested_account, action):
        response = "🚫 Access Denied: You are not authorized"

    else:
        # Step 3: Tool Invocation
        if "balance" in action:
            response = get_balance(requested_account)

        elif "loan" in action:
            response = approve_loan(requested_account)

        # Fallback if LLM fails
        elif action == "fallback":
            msg_lower = message.lower()
            if "balance" in msg_lower:
                response = get_balance(requested_account)
            elif "loan" in msg_lower:
                response = approve_loan(requested_account)
            else:
                response = "⚠️ Could not understand request"

        else:
            response = "🤖 Try asking about balance or loan"

    # Save chat history
    history.append((message, response))

    return "", history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏦 AI Banking Assistant (MCP + RBAC + Groq)")

    role = gr.Dropdown(
        ["customer", "employee", "manager"],
        label="Select Role"
    )

    user_account = gr.Textbox(label="Your Account ID (e.g., 1001)")
    requested_account = gr.Textbox(label="Target Account ID (optional)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, role, user_account, requested_account, state],
        outputs=[msg, chatbot]
    )


# ======================================
# STEP 9: Launch App
# ======================================
demo.launch(share=True)

ModuleNotFoundError: No module named 'google.colab'

In [6]:
# ======================================
# STEP 1: Install Libraries
# ======================================
# !pip install groq nest_asyncio


# ======================================K
# STEP 2: Load Groq API Key (Colab Secret)
# ======================================
# from google.colab import userdata
from dotenv import load_dotenv
load_dotenv()
import os
groq_api_key = os.environ['GROQ_API_KEY']
from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: MOCK TOOLS (Simulating MCP Tools)
# ======================================

import asyncio
import random
import nest_asyncio

# Apply nest_asyncio to allow nested event loops
nest_asyncio.apply()

async def web_search(query):
    await asyncio.sleep(1)  # simulate delay
    return f"📰 News about {query}: Market is growing fast."

async def get_stock_data(company):
    await asyncio.sleep(1)
    price = random.randint(100, 500)
    return f"📈 Stock price of {company}: ${price}"

async def fetch_company_profile(company):
    await asyncio.sleep(1)
    return f"👥 {company} has 5000 employees, HQ in USA"


# ======================================
# STEP 4: PARALLEL TOOL INVOCATION
# ======================================

async def parallel_research(company):

    results = await asyncio.gather(
        web_search(company),
        get_stock_data(company),
        fetch_company_profile(company),
        return_exceptions=True
    )

    news, stock, profile = results

    return {
        "news": news if not isinstance(news, Exception) else "News unavailable",
        "stock": stock if not isinstance(stock, Exception) else "Stock unavailable",
        "profile": profile if not isinstance(profile, Exception) else "Profile unavailable"
    }


# ======================================
# STEP 5: CHAINED TOOL INVOCATION USING GROQ
# ======================================

def analyse_text(text):
    response = client.chat.completions.create(
    model="llama-3.3-70b-versatile", # Updated model name to a currently available Groq model
        messages=[{
            "role": "user",
            "content": f"Analyze this data and give key insights:\n{text}"
        }]
    )
    return response.choices[0].message.content


def generate_report(analysis, company):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", # Updated model name to a currently available Groq model
        messages=[{
            "role": "user",
            "content": f"Create a professional report for {company}:\n{analysis}"
        }]
    )
    return response.choices[0].message.content


# ======================================
# STEP 6: FULL MCP PIPELINE
# ======================================

async def full_pipeline(company):

    # Step 1: Parallel Data Collection
    data = await parallel_research(company)

    combined_text = f"""
    News: {data['news']}
    Stock: {data['stock']}
    Profile: {data['profile']}
    """

    # Step 2: Analysis (LLM)
    analysis = analyse_text(combined_text)

    # Step 3: Report Generation (LLM)
    report = generate_report(analysis, company)

    return report


# ======================================
# STEP 7: RUN THE SYSTEM
# ======================================

company_name = "nvidia"

# Use asyncio.run() after applying nest_asyncio
result = asyncio.run(full_pipeline(company_name))

from IPython.display import Markdown, display

# print("📊 FINAL REPORT:\n")
display(Markdown("📊 FINAL REPORT:\n"))
display(Markdown(result))

📊 FINAL REPORT:


**NVIDIA Corporation: Market Analysis and Growth Opportunities Report**

**Executive Summary:**

This report provides an analysis of NVIDIA Corporation's current market position, growth prospects, and potential opportunities for expansion. Based on the provided data, our key findings indicate a growing market, a strong stock performance, and a relatively modest company size, which suggests opportunities for growth and investment. However, to gain a more comprehensive understanding of NVIDIA's position, we recommend analyzing additional data, including historical stock prices, financial reports, industry trends, and product innovation.

**Market Analysis:**

1. **Market Growth:** The market is experiencing rapid growth, which is expected to drive increased demand for NVIDIA's products and services. This trend is likely to contribute to revenue growth and improved profitability.
2. **Stock Performance:** With a current stock price of $444, NVIDIA's stock performance appears strong. However, a comprehensive analysis of historical data is necessary to determine whether the stock is overvalued, undervalued, or fairly valued.

**Company Overview:**

1. **Company Size and Location:** NVIDIA has a workforce of 5,000 employees, indicating a sizable company with significant resources. Headquartered in the USA, NVIDIA has access to a large market, skilled workforce, and favorable business environment. However, the company is also subject to US regulations and economic conditions.
2. **Potential for Expansion:** With a growing market and relatively modest workforce, NVIDIA may have opportunities for expansion, including increasing its workforce, entering new markets, or developing new products.

**Investment Opportunity:**

Considering the growing market and company size, NVIDIA may be an attractive investment opportunity for those looking to capitalize on the growth of the tech industry. However, investors should conduct thorough research and consider various factors, including:

1. **Financial Reports:** Revenue, profitability, and other financial metrics to assess the company's financial health and growth prospects.
2. **Industry Trends and Competition:** Analysis of industry trends, competitors, and market share to understand NVIDIA's position and potential for growth.
3. **Product Portfolio and Innovation Pipeline:** Evaluation of NVIDIA's product offerings, research and development initiatives, and innovation pipeline to assess the company's potential for future growth and competitiveness.
4. **Geographic Revenue Breakdown and Expansion Plans:** Analysis of NVIDIA's revenue distribution across different regions and potential expansion plans to identify opportunities for growth and diversification.

**Recommendations:**

1. **Conduct Thorough Research:** Investors and stakeholders should conduct comprehensive research, including analysis of historical data, financial reports, and industry trends, to gain a deeper understanding of NVIDIA's position and potential.
2. **Monitor Market Trends:** Continuously monitor market trends, industry developments, and competitors to stay informed about NVIDIA's growth prospects and potential challenges.
3. **Diversification:** Consider diversifying investments to minimize risk and capitalize on growth opportunities in the tech industry.

**Conclusion:**

NVIDIA Corporation appears to be well-positioned for growth, with a strong market trend, solid stock performance, and opportunities for expansion. However, to make informed investment decisions, it is essential to conduct thorough research and analyze additional data. By doing so, investors and stakeholders can gain a comprehensive understanding of NVIDIA's potential and make informed decisions about their investment strategies.